In [ ]:
!pip install -q google-genai

from google import genai
from google.colab import userdata
import os

# Load API key from Colab Secrets - never hardcode
API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = API_KEY

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello"
)
print(response.text)

In [ ]:
from pydantic import BaseModel

class Invoice(BaseModel):
    invoice_number: str
    date: str
    total_amount: float

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Extract: Invoice #INV-001 dated 2026-01-15 for $450.00",
    config={
        "response_mime_type": "application/json",
        "response_schema": Invoice,
    },
)
print(response.text)

In [ ]:
messy_text = """
Hi, please pay invoice INV-002.
Date: Jan 16, 2026
Amount due: $1,250.50
Thanks!
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"Extract the invoice data from this text: {messy_text}",
    config={
        "response_mime_type": "application/json",
        "response_schema": Invoice,
    },
)
print(response.text)

In [ ]:
from pydantic import BaseModel
from typing import List

class Invoice(BaseModel):
    invoice_number: str
    date: str
    total_amount: float

class InvoiceList(BaseModel):
    invoices: List[Invoice]

bulk_text = """
First invoice: INV-003 dated Feb 1, 2026 for $99.99
Second one is INV-004 from Feb 2, 2026, amount $75.00
Also INV-005 for Feb 3, 2026 total $300.25
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"Extract ALL invoices from this text: {bulk_text}",
    config={
        "response_mime_type": "application/json",
        "response_schema": InvoiceList,
    },
)
print(response.text)

In [ ]:
import pandas as pd
import json

# Convert the JSON response to a Python dict
data = json.loads(response.text)

# Convert to DataFrame and save CSV
df = pd.DataFrame(data['invoices'])
df.to_csv('extracted_invoices.csv', index=False)

print("Saved to CSV! Here's the table:")
print(df)